# Experimento 4. Ablación y curvas de aprendizaje

¿De dónde viene la información de la inversión, y qué pasa si se quita o se degrada una entrada? Resultados de `nox-ml diagnose configs/experiments/baseline.yaml` (línea base: pluma gaussiana, noviembre–diciembre de 2023). Si `runs/baseline/diagnostics/` no existe, el cuaderno lo calcula (unos 8 minutos) y lo deja para la siguiente vez.

Los casos de la ablación, con las mismas métricas que `nox-ml run`:

- **sin TEMPO**: se quitan los píxeles del satélite;
- **24, 15 y 9 estaciones**: subconjuntos al azar pero repartidos por el dominio (5 semillas; 24 es la línea base);
- **viento girado +15° y +30°**: el viento de cada hora se gira en el sentido del reloj, un error sistemático de dirección (la incertidumbre de 15° de la inversión no cambia).

In [ ]:
from nox_ml.validation import report
from nox_ml.validation import report_experiments as rx

RUNS = report.find_runs()
diagnostics = rx.diagnostics_for(RUNS.parent / "configs" / "experiments" / "baseline.yaml", RUNS)
summary = rx.ablation_summary(diagnostics["ablation"])
summary.round(2)

In [ ]:
rx.plot_ablation(summary)

La línea punteada roja es la línea base; las barras de error son la desviación estándar entre los subconjuntos al azar (cero cuando el caso corre una sola vez).

**Lectura.**

- **TEMPO.** Sin satélite el error con estación oculta sube de 13.6 a 14.6 ppb, pero lo importante es el **cociente domingo ÷ hábil: pasa de 0.42 a 0.80**. Sin TEMPO, el fondo horario de las estaciones se intercambia con la emisión y el cambio de emisión casi desaparece. Es la razón por la que el satélite es parte del método y no un adorno.
- **Estaciones.** Con 15 estaciones el error sube a 13.9 ± 0.5 ppb y con 9, a 15.3 ± 1.0. El elegir qué estaciones se usan cambia el error en ±0.5–1 ppb. El cociente domingo ÷ hábil casi no se mueve (0.42): lo aporta TEMPO, no el número de estaciones.
- **Viento girado.** Girar el viento 15° o 30° **no empeora** el error (13.5 y 13.3 ppb; 13.6 en la base) ni el cociente domingo ÷ hábil (0.41 y 0.40) ni la correlación con TEMPO (0.74). Un error sistemático de dirección de 30° no se nota en estas métricas: la dirección del viento **no es lo que limita** hoy al modelo (cuaderno 5).

## Curvas de aprendizaje

Error de la muestra (modelo ajustado con todas las estaciones) y con estación oculta, con menos días (bloques de días consecutivos) y con menos estaciones; también el de IDW, la interpolación de las demás estaciones. Media y desviación estándar entre 5 subconjuntos al azar.

In [ ]:
rx.learning_summary(diagnostics["learning_curve"]).round(2)

In [ ]:
rx.plot_learning_curves(diagnostics["learning_curve"])

**Lectura.**

- **Más días ya no ayudan.** El error con estación oculta baja de 14.1 ppb (15 días) a 13.3 (30 días) y no mejora después (13.6 con los 61). Con 30 días la inversión ya extrajo lo que puede de este modelo.
- **Más estaciones sí ayudan**: de 15.3 ppb con 6 a 13.6 con 24, y IDW mejora aún más rápido (de 14.7 a 11.6). Con las 24 estaciones, IDW le gana al modelo (11.6 contra 13.6 ppb) en concentración: el valor del modelo no está en predecir NOₓ en una estación, sino en estimar **emisiones**.
- **No hay sobreajuste.** El error en la muestra es casi constante (unos 13 ppb) y la brecha con el de estación oculta es de 0.6 ppb con todos los datos. Que el error de entrenamiento no baje indica que el límite es el modelo (transporte, viento, a priori), no la cantidad de datos.

## ¿Quién aporta la información?

Grados de libertad para la señal (de 31 incógnitas: 4 fuentes puntuales y 9 regiones × 3 tipos de día), con solo las estaciones, solo TEMPO y ambos. Se calculan con el a priori y la matriz, sin usar los valores observados. Los píxeles de TEMPO se tratan como independientes, como en la línea base, así que su aporte es una **cota superior**.

In [ ]:
diagnostics["degrees_of_freedom"].round(2)

**Lectura.** Las estaciones solas dan unos 26 de los 31 grados de libertad y TEMPO los 31 (la cota superior: con píxeles independientes se satura), de modo que «ambos» no suma nada sobre TEMPO. En cada tipo de día, las estaciones dan 7.2–8.5 de 9 y TEMPO 9; en las fuentes puntuales, 3.0 de 4 contra 4.0. Son cotas sobre lo que **podría** saberse; lo que de verdad pasa al quitar el satélite es lo que muestra la ablación (el cociente domingo ÷ hábil sube de 0.42 a 0.80).